# Model Building — Malware Detection

Nesta fase são desenvolvidos e comparados três modelos de classificação supervisionada para deteção de malware.

Os modelos utilizados são:

- Decision Tree;
- Random Forest;
- XGBoost.

O objetivo é treinar os modelos com os dados preparados, gerar previsões sobre o conjunto de teste e comparar o seu desempenho através de métricas adequadas a classificação binária.

In [ ]:
# Preparar Ambiente

import sys
import subprocess
from pathlib import Path

em_colab = "google.colab" in sys.modules

if em_colab:
    pasta_projeto = Path("/content/IAAC-Projeto")

    if not pasta_projeto.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "https://github.com/diogocarvalho2025056-crypto/IAAC-Projeto.git"
            ],
            check=True
        )
else:
    pasta_projeto = Path.cwd().parent

if str(pasta_projeto) not in sys.path:
    sys.path.insert(0, str(pasta_projeto))

print("Pasta do projeto:")
print(pasta_projeto)

In [ ]:
# Instalar Dependências

%pip install -q kagglehub[pandas-datasets] xgboost

In [ ]:
# Imports

import pandas as pd
import kagglehub

from kagglehub import KaggleDatasetAdapter

from src.data_preparation_utils import (
    remover_colunas,
    remover_duplicados,
    separar_features_target,
    dividir_treino_teste
)

from src.model_utils import (
    criar_decision_tree,
    criar_random_forest,
    criar_xgboost,
    treinar_modelo,
    prever,
    avaliar_modelo,
    comparar_modelos,
    comparar_treino_teste,
    importancia_features
)

## Importação do dataset

O dataset é importado diretamente da plataforma Kaggle.

É utilizado o ficheiro combinado `Malware_and_benign_recognition.csv`, contendo exemplos de ficheiros benignos e maliciosos.

In [ ]:
# Importar do Kaggle

file_path = "Malware_and_benign_recognition.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "aashishzzz/malware-detection-dataset-with-new-malware-info",
    file_path
)

print("Dataset importado com sucesso.")
print("Dimensão original:", df.shape)

## Preparação dos dados

São aplicadas as mesmas decisões definidas na fase de Data Preparation, de forma a garantir que o processo de modelação é reprodutível.

São removidas as seguintes colunas:

- `File`;
- `SuspiciousSectionNames`;
- `SectionCount`.

São também removidos os duplicados completos identificados após esta limpeza.


In [ ]:
# Preparação dos dados

df_model = df.copy()

colunas_remover = [
    "File",
    "SuspiciousSectionNames",
    "SectionCount"
]

df_model = remover_colunas(
    df_model,
    colunas_remover
)

df_model = remover_duplicados(
    df_model
)

print("Dimensão após preparação:")
print(df_model.shape)

In [ ]:
# Features e Target

X, y = separar_features_target(
    df_model,
    "Malicious"
)

print("Features:", X.shape)
print("Target:", y.shape)

In [ ]:
# Treino e teste - divisão

X_train, X_test, y_train, y_test = dividir_treino_teste(
    X,
    y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## Modelos utilizados

Foram selecionados três modelos supervisionados de classificação.

### Decision Tree
Utiliza uma estrutura de decisões hierárquicas para separar as classes.

### Random Forest
Combina várias árvores de decisão através de uma abordagem de ensemble baseada em bagging.

### XGBoost
Utiliza boosting, construindo árvores sequencialmente com o objetivo de corrigir erros cometidos pelas anteriores.

Nesta primeira análise são utilizadas configurações iniciais dos modelos, mantendo `random_state=42` para garantir reprodutibilidade. Não é ainda realizada otimização sistemática de hiperparâmetros.

In [ ]:
# Criar Modelos

decision_tree = criar_decision_tree()

random_forest = criar_random_forest()

xgboost = criar_xgboost()

In [ ]:
# Treinar modelos

decision_tree = treinar_modelo(
    decision_tree,
    X_train,
    y_train
)

random_forest = treinar_modelo(
    random_forest,
    X_train,
    y_train
)

xgboost = treinar_modelo(
    xgboost,
    X_train,
    y_train
)

print("Modelos treinados com sucesso.")

In [ ]:
# Previsões

y_pred_dt = prever(
    decision_tree,
    X_test
)

y_pred_rf = prever(
    random_forest,
    X_test
)

y_pred_xgb = prever(
    xgboost,
    X_test
)

## Avaliação dos modelos

Os modelos são avaliados através das seguintes métricas:

- Accuracy;
- Precision;
- Recall;
- F1-score;
- matriz de confusão;
- relatório de classificação.

No contexto deste projeto, o Recall da classe malware é particularmente relevante, porque um falso negativo corresponde a um ficheiro malicioso classificado como benigno.

In [ ]:
# Avaliar Decision Tree

resultado_dt, matriz_dt, relatorio_dt = avaliar_modelo(
    "Decision Tree",
    y_test,
    y_pred_dt,
    ["Benigno", "Malware"]
)

print("--- DECISION TREE ---")
print(resultado_dt)

print("\nMatriz de confusão:")
print(matriz_dt)

print("\nRelatório:")
print(relatorio_dt)

In [ ]:
# Avaliar Random Forest

resultado_rf, matriz_rf, relatorio_rf = avaliar_modelo(
    "Random Forest",
    y_test,
    y_pred_rf,
    ["Benigno", "Malware"]
)

print("--- RANDOM FOREST ---")
print(resultado_rf)

print("\nMatriz de confusão:")
print(matriz_rf)

print("\nRelatório:")
print(relatorio_rf)

In [ ]:
# Avaliar XGBoost

resultado_xgb, matriz_xgb, relatorio_xgb = avaliar_modelo(
    "XGBoost",
    y_test,
    y_pred_xgb,
    ["Benigno", "Malware"]
)

print("--- XGBOOST ---")
print(resultado_xgb)

print("\nMatriz de confusão:")
print(matriz_xgb)

print("\nRelatório:")
print(relatorio_xgb)

In [ ]:
# Comparação Geral

resultados = comparar_modelos([
    resultado_dt,
    resultado_rf,
    resultado_xgb
])

display(resultados)

### Comparação inicial

A Decision Tree apresentou um desempenho global elevado, mas inferior aos dois modelos ensemble.

Random Forest e XGBoost obtiveram resultados idênticos neste conjunto de teste, com Accuracy e F1-score próximos de 97,4%.

O Recall da classe malware foi especialmente elevado em Random Forest e XGBoost, com apenas um falso negativo em 113 exemplos maliciosos presentes no conjunto de teste.

In [ ]:
# Treino VS Teste

treino_dt, teste_dt = comparar_treino_teste(
    decision_tree,
    X_train,
    y_train,
    X_test,
    y_test
)

treino_rf, teste_rf = comparar_treino_teste(
    random_forest,
    X_train,
    y_train,
    X_test,
    y_test
)

treino_xgb, teste_xgb = comparar_treino_teste(
    xgboost,
    X_train,
    y_train,
    X_test,
    y_test
)

comparacao_overfitting = pd.DataFrame({
    "Modelo": [
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy treino": [
        treino_dt,
        treino_rf,
        treino_xgb
    ],
    "Accuracy teste": [
        teste_dt,
        teste_rf,
        teste_xgb
    ]
})

display(comparacao_overfitting)

### Treino vs teste

Os três modelos atingiram Accuracy de 100% no conjunto de treino.

No conjunto de teste, a Accuracy foi inferior, sobretudo na Decision Tree.

Esta diferença pode indicar algum nível de overfitting, especialmente no modelo de árvore única.

Nesta fase estes resultados são mantidos como baseline, não sendo ainda realizada otimização de hiperparâmetros.

## Análise da importância das features

Os três modelos utilizados permitem estimar a importância relativa das características utilizadas na classificação.

Esta análise ajuda a perceber quais as variáveis que mais contribuíram para as decisões dos modelos.

Os valores de importância devem, no entanto, ser interpretados com cautela, uma vez que representam a utilização das variáveis pelo modelo e não uma relação causal com a presença de malware.

In [ ]:
# Feature importance - Decision Tree

importancias_dt = importancia_features(
    decision_tree,
    X.columns
)

display(importancias_dt.head(10))

In [ ]:
# Feature importance - Random Forest

importancias_rf = importancia_features(
    random_forest,
    X.columns
)

display(importancias_rf.head(10))

In [ ]:
# Feature importance - XGBoost

importancias_xgb = importancia_features(
    xgboost,
    X.columns
)

display(importancias_xgb.head(10))

## Importância das features

A análise das feature importances mostra que algumas variáveis já identificadas durante o Data Understanding voltam a surgir com peso significativo nos modelos.

Entre as características mais relevantes encontram-se:

- `ImageBase`;
- `SizeOfHeaders`;
- `BaseOfData`;
- `DllCharacteristics`;
- `Characteristics`.

A distribuição da importância não é igual entre os modelos.

A Decision Tree concentra grande parte da decisão em poucas características, enquanto a Random Forest distribui a importância por um conjunto mais alargado de variáveis.

Estas importâncias representam a forma como os modelos utilizam as características para realizar as previsões e não devem ser interpretadas como relações causais.

## Conclusão

Foram desenvolvidos três modelos supervisionados para classificação de ficheiros como benignos ou maliciosos: Decision Tree, Random Forest e XGBoost.

Todos os modelos apresentaram desempenho elevado no conjunto de teste.

A Decision Tree atingiu uma Accuracy próxima de 95,2%, enquanto Random Forest e XGBoost apresentaram resultados idênticos neste conjunto de teste, atingindo aproximadamente 97,4% de Accuracy.

Random Forest e XGBoost apresentaram também um Recall de aproximadamente 99,1% para a classe malware, correspondendo a apenas um falso negativo no conjunto de teste.

Os três modelos atingiram 100% de Accuracy nos dados de treino, o que justifica uma análise posterior de possível overfitting, sobretudo tendo em conta a diferença entre o desempenho no treino e no teste.

Estes resultados representam modelos baseline. Numa fase posterior poderão ser aplicadas técnicas de validação cruzada e otimização de hiperparâmetros para avaliar a estabilidade dos modelos e melhorar a sua capacidade de generalização.